In [ ]:
import cv2
import numpy as np

cap = cv2.VideoCapture('./video2_3sec.mp4')

numMaxCorners = 20
feature_params = dict(maxCorners = numMaxCorners,
                      qualityLevel=0.3,
                      minDistance=7,
                      blockSize=7)

lk_params = dict(winSize=(15, 15),
                 maxLevel=2,
                 criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

ret, prevFrame = cap.read()
prevFrame = cv2.resize(prevFrame, (320, 240))
prevGray = cv2.cvtColor(prevFrame, cv2.COLOR_BGR2GRAY)
p0 = cv2.goodFeaturesToTrack(prevGray, mask = None, **feature_params)

flowColor = np.random.randint(0, 255, (numMaxCorners, 3))
mask = np.zeros_like(prevFrame)
while(1):
    ret, curFrame = cap.read()
    if ret != True:
        break
    curFrame = cv2.resize(curFrame, (320, 240))
    curGray = cv2.cvtColor(curFrame, cv2.COLOR_BGR2GRAY)

    p1, st, err = cv2.calcOpticalFlowPyrLK(prevGray, curGray, p0, None, **lk_params)

    good_new = p1[st==1]
    good_old = p0[st==1]

    prevGray = curGray.copy()
    p0 = good_new.reshape(-1, 1, 2)

    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a, b = new.ravel()
        c, d = old.ravel()
        mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), flowColor[i].tolist(), 2)
        curFrame = cv2.circle(curFrame, (int(a), int(b)), 5, flowColor[i].tolist(), -1)
    res = cv2.add(curFrame, mask)
    cv2.imshow('frame', res)

    k = cv2.waitKey(30) & 0xff
    if k == 27:
        break

: 

In [1]:
import cv2
import numpy as np

cap = cv2.VideoCapture('./video2_3sec.mp4')

ret, prevFrame = cap.read()
prevFrame = cv2.resize(prevFrame, (320, 240))
prvs = cv2.cvtColor(prevFrame, cv2.COLOR_BGR2GRAY)

maskHSV = np.zeros_like(prevFrame)
maskHSV[...,1]=255

fb_params = dict(pyr_scale=0.5,
                 levels=3,
                 winsize=15,
                 iterations=3,
                 poly_n=5,
                 poly_sigma=1.2,
                 flags=0)

while(1):
    ret, curFrame = cap.read()
    if ret != True:
        break
    curFrame = cv2.resize(curFrame, (320, 240))
    next = cv2.cvtColor(curFrame, cv2.COLOR_BGR2GRAY)

    flow = cv2.calcOpticalFlowFarneback(prvs, next, None, **fb_params)

    prvs = next

    mag, ang = cv2.cartToPolar(flow[...,0], flow[...,1])
    maskHSV[...,0] = ang*180 / np.pi / 2
    maskHSV[...,2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
    res = cv2.cvtColor(maskHSV, cv2.COLOR_HSV2BGR)
    cv2.imshow('frame2', res)

    k = cv2.waitKey(30) & 0xff
    if k == 27:
        break
cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import numpy as np

tracker_types = ['MIL', 'KCF', 'TLD', 'MEDIANFLOW', 'GOTURN', 'MOSSE', 'CSRT']
for tracker_type in tracker_types:
    # if tracker_type == 'BOOSTING':
    #     tracker = cv2.TrackerBoosting_create()
    if tracker_type == 'MIL':
        tracker = cv2.TrackerMIL_create()
    if tracker_type == 'KCF':
        tracker = cv2.TrackerKCF_create()
    if tracker_type == 'TLD':
        tracker = cv2.TrackerTLD_create()
    if tracker_type == 'MEDIANFLOW':
        tracker = cv2.TrackerMedianFlow_create()
    if tracker_type == 'GOTURN':
        tracker = cv2.TrackerGOTURN_create()
    if tracker_type == 'MOSSE':
        tracker = cv2.TrackerMOSSE_create()
    if tracker_type == 'CSRT':
        tracker = cv2.TrackerCSRT_create()
    
    cap = cv2.VideoCapture('./video4.mp4')

    ret, frame = cap.read()
    if not ret:
        exit(-1)
    frame = cv2.resize(frame, (320, 240))
    bbox = cv2.selectROI(frame, False)
    while bbox == (0, 0, 0, 0):
        bbox = cv2.selectROI(frame, False)
    
    ok = tracker.init(frame, bbox)

    while True:
        ret, frame = cap.read()
        if not ret:
            cv2.destroyAllWindows()
            break

        timer = cv2.getTickCount()

        frame = cv2.resize(frame, (320, 240))
        ret_tracker, bbox = tracker.update(frame)
        
        fps = cv2.getTickFrequency() / (cv2.getTickCount() - timer)

        if ret_tracker:
            p1 = (int(bbox[0]), int(bbox[1]))
            p2 = (int(bbox[0] + bbox[2]), int(bbox[1] + bbox[3]))
            cv2.rectangle(frame, p1, p2, (0, 0, 255), 2, 1)
        else:
            cv2.putText(frame, 'Tracking failure', (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
        
        cv2.putText(frame, tracker_type + 'Tracker', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
        cv2.putText(frame, 'FPS : ' + str(int(fps)), (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
        cv2.imshow('Tracking', frame)

        k = cv2.waitKey(1) & 0xff
        if k == 27:
            break

error: OpenCV(4.12.0) D:\a\opencv-python\opencv-python\opencv\modules\core\src\merge.dispatch.cpp:121: error: (-215:Assertion failed) !mv[0].empty() in function 'cv::merge'


: 